# 3 · Den Shift selbst messen – ohne Labels

**Frage:** Wie verschieden sind die Spitäler, und verringert Normalisierung diesen Unterschied? Diese Messungen brauchen keine Diagnose-Labels und lassen sich deshalb auch auf neue Daten anwenden, bevor man ein Modell dort einsetzt.

**Drei Ebenen**
1. **Farbe:** Lab-Statistik der Gewebepixel pro Spital.
2. **Färbung:** Macenko-Farbvektoren pro Slide.
3. **Merkmale:** Wie unterscheiden sich die Spitäler im Merkmalsraum eines Netzes? Gemessen mit
   - **Herkunfts-Vorhersagbarkeit:** Lässt sich das Spital aus den Merkmalen erraten? Zufall = 20 %.
   - **MMD** (Maximum Mean Discrepancy) zwischen je zwei Spitälern.

Alles wird für rohe und Macenko-normalisierte Patches berechnet.

**Voraussetzung:** Notebook 1 (für `results/stain_params.pkl`; optional das trainierte Baseline-Modell).

In [ ]:
# Einmalig installieren (oder: pip install -r requirements.txt)
# %pip install -r requirements.txt

import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import stainlib as sl
import trainlib as tl

DEVICE = tl.get_device()
print("Gerät:", DEVICE)

## Konfiguration und Stichprobe

In [ ]:
DATA_ROOT = "data"
N_PER_HOSP = 600
FEATURES_FROM = "imagenet"   # "imagenet" = neutrales vortrainiertes Netz | "baseline" = Modell aus Notebook 1
WORKERS = 4
# Macenko-Filter für dunkle Pixel: None = ursprünglicher Lauf, 60 = korrigiert (braucht stain_params_dark60.pkl)
MACENKO_DARK = 60
SUF = "" if MACENKO_DARK is None else ("_dark" if MACENKO_DARK == 30 else f"_dark{MACENKO_DARK}")

wds, meta = tl.load_camelyon(DATA_ROOT, download=False)
sp = tl.load_pickle(f"results/stain_params{SUF}.pkl")
ref, slide_params = sp["ref"], sp["slide_params"]

rng = np.random.default_rng(0)
idx = np.concatenate([tl._balanced(np.where(meta["hospital"] == h)[0], meta["y"], N_PER_HOSP, rng) for h in range(5)])
hosp = meta["hospital"][idx]
print("Macenko-Filter:", MACENKO_DARK, "| Patches pro Spital:", np.bincount(hosp))

## Ebene 1 · Farbe

Pro Patch Mittelwerte von L* (Helligkeit), a* (Grün–Rot) und b* (Blau–Gelb) über die Gewebepixel. Ein Mass für den Shift: die Streuung der Spital-Mittelwerte. Wird sie durch Normalisierung kleiner, gleicht die Normalisierung die Spitäler an.

In [ ]:
rows = []
for i in idx:
    img = tl.load_image(wds, i)
    p = slide_params.get(int(meta["slide"][i]))
    rows.append({"hospital": int(meta["hospital"][i]), "version": "roh", **sl.lab_stats(img)})
    rows.append({"hospital": int(meta["hospital"][i]), "version": "Macenko",
                 **sl.lab_stats(sl.macenko_apply(img, p, ref) if p else img)})
color = pd.DataFrame(rows)
color.to_csv(f"results/nb3_color_stats{SUF}.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, ch in zip(ax, ["L_mean", "a_mean", "b_mean"]):
    data, labels = [], []
    for h in range(5):
        for ver in ["roh", "Macenko"]:
            data.append(color[(color.hospital == h) & (color.version == ver)][ch]); labels.append(f"{h}\n{ver[:3]}")
    bp = a.boxplot(data, showfliers=False, patch_artist=True)
    a.set_xticks(range(1, len(labels) + 1), labels, fontsize=8)
    for k, box in enumerate(bp["boxes"]):
        box.set_facecolor("#c9d4f2" if k % 2 == 0 else "#f6d3c4")
    a.set_title(ch.replace("_mean", "*")); a.set_xlabel("Spital / Version")
plt.tight_layout(); plt.savefig(f"results/nb3_color_boxplots{SUF}.png", dpi=200); plt.show()

spread = (color.groupby(["version", "hospital"])[["L_mean", "a_mean", "b_mean"]].mean()
          .groupby("version").std().round(2))
print("Streuung der Spital-Mittelwerte (kleiner = einheitlicher):")
spread

# Farbabstand ΔE (CIE76) zwischen den Spital-Mittelwerten: Mittel und Maximum über alle zehn Paare
from itertools import combinations
mu = color.groupby(["version", "hospital"])[["L_mean", "a_mean", "b_mean"]].mean()
for ver in ["roh", "Macenko"]:
    d = {(i, j): float(np.linalg.norm(mu.loc[(ver, i)] - mu.loc[(ver, j)])) for i, j in combinations(range(5), 2)}
    worst = max(d, key=d.get)
    print(f"ΔE {ver:8s}: Mittel {np.mean(list(d.values())):.1f} | Maximum {d[worst]:.1f} (Spital {worst[0]} vs. {worst[1]})")
# Verdächtige Slides in den verwendeten Parametern (H-Winkel < 5° oder H-Intensität > 9)
bad = [s for s, p in slide_params.items() if p.maxC[0] > 9.0 or p.angles_deg()[0] < 5.0]
print(f"Verdächtige Slides: {len(bad)} von {len(slide_params)} {bad}")

## Ebene 2 · Färbung: Farbvektoren pro Slide

In [ ]:
sv = []
for s, p in slide_params.items():
    h = int(np.bincount(meta["hospital"][meta["slide"] == s]).argmax())
    a_h, a_e = p.angles_deg()
    sv.append({"slide": s, "hospital": h, "H_winkel": a_h, "E_winkel": a_e, "H_max": p.maxC[0], "E_max": p.maxC[1]})
sv = pd.DataFrame(sv)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for h, g in sv.groupby("hospital"):
    ax[0].scatter(g["H_winkel"], g["E_winkel"], label=f"Spital {h}", s=40)
    ax[1].scatter(g["H_max"], g["E_max"], label=f"Spital {h}", s=40)
ra, re_ = ref.angles_deg()
ax[0].scatter([ra], [re_], marker="*", s=250, c="k", label="Referenz")
ax[1].scatter([ref.maxC[0]], [ref.maxC[1]], marker="*", s=250, c="k")
ax[0].set_xlabel("H-Vektor: Winkel zur Graudiagonale (°)"); ax[0].set_ylabel("E-Vektor: Winkel (°)")
ax[0].set_title("Farbton der Farbstoffe pro Slide")
ax[1].set_xlabel("H-Intensität (99. Perzentil)"); ax[1].set_ylabel("E-Intensität (99. Perzentil)")
ax[1].set_title("Färbeintensität pro Slide")
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"results/nb3_stain_vectors{SUF}.png", dpi=200); plt.show()
sv.groupby("hospital")[["H_winkel", "E_winkel", "H_max", "E_max"]].mean().round(2)

## Ebene 3 · Merkmalsraum

Standard ist ein ImageNet-vortrainiertes ResNet-50: Es kennt weder Pathologie noch die Spitäler und misst daher den Shift „neutral“. Mit `FEATURES_FROM = "baseline"` misst man stattdessen, wie stark *das eigene Modell* die Spitäler trennt – das ist näher an der Frage, ob es durch den Shift gestört wird.

Für MMD werden die Merkmale zuerst per PCA auf 50 Dimensionen reduziert.

In [ ]:
from sklearn.decomposition import PCA

if FEATURES_FROM == "imagenet":
    fmodel = tl.build_model("resnet50", pretrained=True)
else:
    split = tl.load_json("results/split.json"); tag = "-".join(map(str, split["train_hosp"]))
    fmodel = tl.build_model("resnet18", pretrained=False)
    fmodel.load_state_dict(tl.torch.load(f"models/baseline_seed0_train{tag}.pt", map_location="cpu"))

feats = {}
for ver, mode in [("roh", "baseline"), ("Macenko", "macenko")]:
    ds = tl.PatchDataset(wds, idx, meta, mode=mode, slide_params=slide_params, ref=ref)
    feats[ver] = tl.extract_features(fmodel, ds, workers=WORKERS, device=DEVICE)
print({k: v.shape for k, v in feats.items()})

In [ ]:
summary, mmd = [], {}
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (ver, F) in zip(ax, feats.items()):
    Z = PCA(n_components=50, random_state=0).fit_transform(F)
    pred = sl.origin_predictability(Z, hosp)
    keys, M_ = sl.pairwise_matrix({h: Z[hosp == h] for h in range(5)}, sl.mmd_rbf)
    mmd[ver] = pd.DataFrame(M_, index=keys, columns=keys)
    summary.append({"version": ver, "Spital erkennbar (bal. Acc.)": pred["balanced_accuracy"],
                    "Zufall": pred["chance"], "mittlere MMD": M_[np.triu_indices(5, 1)].mean()})
    for h in range(5):
        a.scatter(Z[hosp == h, 0], Z[hosp == h, 1], s=5, alpha=0.5, label=f"Spital {h}")
    a.set_title(f"{ver}: Spital zu {pred['balanced_accuracy']:.0%} erkennbar (Zufall 20 %)")
    a.set_xlabel("PC 1"); a.set_ylabel("PC 2")
ax[0].legend(markerscale=3, fontsize=8)
plt.tight_layout(); plt.savefig(f"results/nb3_feature_pca{SUF}.png", dpi=200); plt.show()
summary = pd.DataFrame(summary).round(3)
summary.to_csv(f"results/nb3_shift_summary{SUF}.csv", index=False)
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
vmax = max(m.values.max() for m in mmd.values())
for a, (ver, m) in zip(ax, mmd.items()):
    im = a.imshow(m.values, vmin=0, vmax=vmax, cmap="magma_r")
    a.set_xticks(range(5), m.columns); a.set_yticks(range(5), m.index)
    a.set_xlabel("Spital"); a.set_ylabel("Spital"); a.set_title(f"MMD zwischen Spitälern ({ver})")
    for (r_, c_), val in np.ndenumerate(m.values):
        a.text(c_, r_, f"{val:.3f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.savefig(f"results/nb3_mmd{SUF}.png", dpi=200, bbox_inches="tight"); plt.show()

## Optional: Shift und Leistungsverlust in Beziehung setzen

Braucht das vollständige Leave-one-center-out aus Notebook 1 (`results/nb1_loco.csv`). Für jedes zurückgehaltene Spital: mittlere MMD zu den vier Trainingsspitälern gegen den Genauigkeitsverlust des Baseline-Modells. Mit nur fünf Punkten ist das eine Illustration, kein statistischer Nachweis.

In [ ]:
p = Path("results/nb1_loco.csv")
if p.exists():
    loco = pd.read_csv(p)
    b = loco[loco["variant"] == "baseline"]
    pts = []
    for h in range(5):
        g = b[b["held_out"] == h]
        drop = g[g["eval"] == "id_val"]["acc"].mean() - g[g["eval"] == f"ood_h{h}"]["acc"].mean()
        d = mmd["roh"].loc[h, [k for k in range(5) if k != h]].mean()
        pts.append({"spital": h, "mmd_zu_training": d, "genauigkeitsverlust": drop})
    pts = pd.DataFrame(pts)
    ax = pts.plot.scatter("mmd_zu_training", "genauigkeitsverlust", s=60, figsize=(5, 4))
    for _, r_ in pts.iterrows():
        ax.annotate(f"Spital {int(r_.spital)}", (r_.mmd_zu_training, r_.genauigkeitsverlust), xytext=(5, 3),
                    textcoords="offset points")
    ax.set_title("Grösserer Shift → grösserer Verlust?"); plt.tight_layout(); plt.show()
    display(pts.round(3))
else:
    print("results/nb1_loco.csv fehlt – in Notebook 1 RUN_FULL_LOCO = True setzen.")

## So präsentierst du das

- **Ein Satz pro Ebene:** „Die Spitäler unterscheiden sich im Farbton um … (Lab), in der H-Intensität um … – nach Normalisierung schrumpft die Streuung auf …“
- **Die stärkste Zahl:** „Aus den Bildmerkmalen lässt sich das Spital zu X % erraten (Zufall: 20 %). Nach Macenko-Normalisierung noch zu Y %.“ Das macht den Batch-Effekt greifbar.
- **Grenze:** Bleibt das Spital nach der Normalisierung gut erkennbar, unterscheiden sich die Daten nicht nur in der Farbe (Scanner-Schärfe, Gewebeaufbereitung, Patientenpopulation).
- **Weiterführend:** Mit einem Pathologie-Foundation-Model (z. B. UNI, Virchow; Zugang über Hugging Face mit Nutzungsbedingungen) statt ResNet-50 als Merkmalsextraktor lässt sich prüfen, wie robust diese Modelle tatsächlich sind.